# GNC-X (4-Phase X-Folding) — Benchmark de Gargalos de Memória em GPU (Google Colab Tesla T4)
**Autor da Concepção:** Zaqueu Ribeiro da Costa

Este notebook compila e executa os kernels CUDA (`sm_75`) da arquitetura **GNC-X** diretamente na GPU **NVIDIA Tesla T4** do Google Colab, avaliando:
1. **Gargalo PCIe Host-to-Device (`H2D`):** `int64` Pageable (8B) vs. `CompactXToken32` Pinned (4B) vs. `PackedXToken16` (2B).
2. **Coalescência de Warp (128 bytes):** Leitura empacotada em `uint32_t`.
3. **Contenção Atômica nos 10 Crivos Diagonais:** `K1 (Naive Global Atomic)` vs `K2 (Shared Atomic)` vs `K3 (GNC-X Register + __shfl_down_sync + Padded Shared [8][16])`.


In [ ]:
%%writefile gncx_t4_colab.cu

// ============================================================================
// GNC-X CUDA C++ KERNELS FOR NVIDIA TESLA T4 (Turing sm_75)
// Compile on Colab T4: nvcc -O3 -arch=sm_75 gncx_t4_benchmark.cu -o gncx_t4_benchmark
// ============================================================================
#include <cuda_runtime.h>
#include <cstdint>
#include <cstdio>
#include <vector>
#include <algorithm>

__constant__ int64_t d_DIAG_ANCHORS_L3[10] = {
    0LL, 1111LL, 2222LL, 3333LL, 4444LL, 5555LL, 6666LL, 7777LL, 8888LL, 9999LL
};

// Packed 32-bit token for 100% coalesced 128-byte Warp transactions on GDDR6:
// bits [1:0]   = phase_2bit (00=--, 01=-+, 10=+-, 11=++)
// bits [15:8]  = diag_d     (0..9)
// bits [31:16] = abs_delta  (uint16_t)
using PackedXToken32 = uint32_t;

// ----------------------------------------------------------------------------
// KERNEL 0: Dense Brute-Force Baseline on GPU (N * M multiplications)
// ----------------------------------------------------------------------------
__global__ void kernel_dense_baseline(
    const PackedXToken32* __restrict__ tokens,
    size_t n,
    const int64_t* __restrict__ weights_m_by_10,
    size_t out_channels,
    unsigned long long* __restrict__ out_channels_acc
) {
    size_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= n) return;

    uint32_t raw = tokens[idx];
    uint32_t phase = raw & 0x3U;
    uint32_t d = (raw >> 8) & 0xFFU;
    int64_t abs_delta = static_cast<int64_t>(raw >> 16);

    int64_t b1 = static_cast<int64_t>((phase >> 1) & 1U);
    int64_t b0 = static_cast<int64_t>(phase & 1U);
    int64_t s1 = (b1 << 1) - 1;
    int64_t s0 = (b0 << 1) - 1;
    int64_t val = s1 * d_DIAG_ANCHORS_L3[d] + s0 * abs_delta;

    for (size_t m = 0; m < out_channels; ++m) {
        int64_t prod = weights_m_by_10[m * 10 + d] * val;
        atomicAdd(&out_channels_acc[m], static_cast<unsigned long long>(prod));
    }
}

// ----------------------------------------------------------------------------
// KERNEL 1 (BOTTLENECK DEMO): Naive Global Atomic X-Fold
// All threads hammer the same 10 global memory addresses `global_sector_mass[0..9]`!
// Demonstrates massive L2 atomic serialization across all 40 SMs of the T4.
// ----------------------------------------------------------------------------
__global__ void kernel_xfold_naive_global_atomic(
    const PackedXToken32* __restrict__ tokens,
    size_t n,
    unsigned long long* __restrict__ global_sector_mass
) {
    size_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= n) return;

    uint32_t raw = tokens[idx];
    uint32_t phase = raw & 0x3U;
    uint32_t d = (raw >> 8) & 0xFFU;
    int64_t abs_delta = static_cast<int64_t>(raw >> 16);

    int64_t b1 = static_cast<int64_t>((phase >> 1) & 1U);
    int64_t b0 = static_cast<int64_t>(phase & 1U);
    int64_t s1 = (b1 << 1) - 1;
    int64_t s0 = (b0 << 1) - 1;
    int64_t val = s1 * d_DIAG_ANCHORS_L3[d] + s0 * abs_delta;

    // Bottleneck: 2,560+ concurrent threads colliding on 10 global addresses!
    atomicAdd(&global_sector_mass[d], static_cast<unsigned long long>(val));
}

// ----------------------------------------------------------------------------
// KERNEL 2: Shared Memory X-Fold (Unpadded 10-bin shared array)
// Reduces global traffic, but still suffers intra-block shared atomic contention.
// ----------------------------------------------------------------------------
__global__ void kernel_xfold_shared_atomic(
    const PackedXToken32* __restrict__ tokens,
    size_t n,
    unsigned long long* __restrict__ global_sector_mass
) {
    __shared__ unsigned long long s_mass[10];
    if (threadIdx.x < 10) s_mass[threadIdx.x] = 0ULL;
    __syncthreads();

    size_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        uint32_t raw = tokens[idx];
        uint32_t phase = raw & 0x3U;
        uint32_t d = (raw >> 8) & 0xFFU;
        int64_t abs_delta = static_cast<int64_t>(raw >> 16);

        int64_t b1 = static_cast<int64_t>((phase >> 1) & 1U);
        int64_t b0 = static_cast<int64_t>(phase & 1U);
        int64_t s1 = (b1 << 1) - 1;
        int64_t s0 = (b0 << 1) - 1;
        int64_t val = s1 * d_DIAG_ANCHORS_L3[d] + s0 * abs_delta;

        atomicAdd(&s_mass[d], static_cast<unsigned long long>(val));
    }
    __syncthreads();

    if (threadIdx.x < 10) {
        atomicAdd(&global_sector_mass[threadIdx.x], s_mass[threadIdx.x]);
    }
}

// ----------------------------------------------------------------------------
// KERNEL 3 (GNC-X OPTIMAL T4 KERNEL):
// Coalesced uint32 Grid-Stride + Register-Privatized 10-Pivot Accumulators
// + Warp-Shuffle (__shfl_down_sync) + Bank-Conflict-Free Padded Shared Memory!
// ZERO atomic collisions inside the inner loop!
// ----------------------------------------------------------------------------
__global__ void kernel_xfold_warp_privatized_optimal(
    const PackedXToken32* __restrict__ tokens,
    size_t n,
    unsigned long long* __restrict__ global_sector_mass
) {
    // Each thread keeps its own 10 Invariant Diagonal accumulators in registers!
    int64_t reg_pivot_phase[10] = {0};
    int64_t reg_residue_sum[10] = {0};

    // Grid-stride loop: 100% coalesced 128-byte Warp loads, ZERO atomic ops in loop!
    size_t stride = blockDim.x * gridDim.x;
    for (size_t idx = blockIdx.x * blockDim.x + threadIdx.x; idx < n; idx += stride) {
        uint32_t raw = tokens[idx];
        uint32_t phase = raw & 0x3U;
        uint32_t d = (raw >> 8) & 0xFFU;
        int64_t abs_delta = static_cast<int64_t>(raw >> 16);

        int64_t b1 = static_cast<int64_t>((phase >> 1) & 1U);
        int64_t b0 = static_cast<int64_t>(phase & 1U);
        int64_t s1 = (b1 << 1) - 1;
        int64_t mask0 = -(b0 ^ 1LL);
        int64_t signed_delta = (abs_delta ^ mask0) - mask0;

        #pragma unroll
        for (int k = 0; k < 10; ++k) {
            int64_t match = -static_cast<int64_t>(d == static_cast<uint32_t>(k));
            reg_pivot_phase[k] += (s1 & match);
            reg_residue_sum[k] += (signed_delta & match);
        }
    }

    // Fold thread's 10 diagonals in registers + Warp Shuffle reduction (__shfl_down_sync)
    // Stride 16 in shared memory avoids 32-bank conflicts across warps!
    __shared__ int64_t warp_bins[8][16];
    int lane = threadIdx.x & 31;
    int warp_id = threadIdx.x >> 5;

    #pragma unroll
    for (int d = 0; d < 10; ++d) {
        int64_t val = reg_pivot_phase[d] * d_DIAG_ANCHORS_L3[d] + reg_residue_sum[d];
        for (int offset = 16; offset > 0; offset >>= 1) {
            val += __shfl_down_sync(0xffffffff, val, offset);
        }
        if (lane == 0) {
            warp_bins[warp_id][d] = val;
        }
    }
    __syncthreads();

    // First 10 threads reduce across the 8 warps of the block and emit 1 atomicAdd per block
    if (threadIdx.x < 10) {
        int64_t block_sum = 0;
        int num_warps = blockDim.x >> 5;
        for (int w = 0; w < num_warps; ++w) {
            block_sum += warp_bins[w][threadIdx.x];
        }
        atomicAdd(&global_sector_mass[threadIdx.x], static_cast<unsigned long long>(block_sum));
    }
}

#include <iostream>
#include <iomanip>

int main() {
    const size_t N = 16000000; // 16 Million X-Tokens (64 MB)
    const size_t M = 64;       // 64 Output Channels (1.024 Billion MACs)

    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);
    std::cout << "====================================================================\n";
    std::cout << " GNC-X CUDA BENCHMARK ON: " << prop.name << " (sm_" << prop.major << prop.minor << ")\n";
    std::cout << "====================================================================\n";

    // 1. Allocate Pageable vs Pinned Host Memory
    int64_t* h_int64_pageable = (int64_t*)malloc(N * sizeof(int64_t));
    PackedXToken32* h_xtok_pinned = nullptr;
    cudaMallocHost(&h_xtok_pinned, N * sizeof(PackedXToken32));

    for (size_t i = 0; i < N; ++i) {
        uint32_t d = (i % 9) + 1;
        uint32_t delta = (i * 17) % 400;
        uint32_t phase = i & 0x3U; // 00, 01, 10, 11 balanced
        h_xtok_pinned[i] = (delta << 16) | (d << 8) | phase;
        h_int64_pageable[i] = static_cast<int64_t>(d * 1111 + delta);
    }

    int64_t* d_int64 = nullptr;
    PackedXToken32* d_xtok = nullptr;
    unsigned long long* d_sector_mass = nullptr;
    unsigned long long* d_out_ch = nullptr;
    int64_t* d_weights = nullptr;

    cudaMalloc(&d_int64, N * sizeof(int64_t));
    cudaMalloc(&d_xtok, N * sizeof(PackedXToken32));
    cudaMalloc(&d_sector_mass, 10 * sizeof(unsigned long long));
    cudaMalloc(&d_out_ch, M * sizeof(unsigned long long));
    cudaMalloc(&d_weights, M * 10 * sizeof(int64_t));

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    auto measure_ms = [&](auto fn) {
        cudaEventRecord(start);
        fn();
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);
        float ms = 0.0f;
        cudaEventElapsedTime(&ms, start, stop);
        return ms;
    };

    float h2d_int64_ms = measure_ms([&]() {
        cudaMemcpy(d_int64, h_int64_pageable, N * sizeof(int64_t), cudaMemcpyHostToDevice);
    });
    float h2d_xtok32_ms = measure_ms([&]() {
        cudaMemcpy(d_xtok, h_xtok_pinned, N * sizeof(PackedXToken32), cudaMemcpyHostToDevice);
    });

    int threads = 256;
    int blocks_full = (N + threads - 1) / threads;
    int blocks_gridstride = 40 * 8; // 40 SMs * 8 blocks/SM on Tesla T4

    // Warmup
    kernel_xfold_warp_privatized_optimal<<<blocks_gridstride, threads>>>(d_xtok, N, d_sector_mass);
    cudaDeviceSynchronize();

    float k0_ms = measure_ms([&]() {
        kernel_dense_baseline<<<blocks_full, threads>>>(d_xtok, N, d_weights, M, d_out_ch);
    });
    float k1_ms = measure_ms([&]() {
        cudaMemsetAsync(d_sector_mass, 0, 10 * sizeof(unsigned long long));
        kernel_xfold_naive_global_atomic<<<blocks_full, threads>>>(d_xtok, N, d_sector_mass);
    });
    float k2_ms = measure_ms([&]() {
        cudaMemsetAsync(d_sector_mass, 0, 10 * sizeof(unsigned long long));
        kernel_xfold_shared_atomic<<<blocks_full, threads>>>(d_xtok, N, d_sector_mass);
    });
    float k3_ms = measure_ms([&]() {
        cudaMemsetAsync(d_sector_mass, 0, 10 * sizeof(unsigned long long));
        kernel_xfold_warp_privatized_optimal<<<blocks_gridstride, threads>>>(d_xtok, N, d_sector_mass);
    });

    std::cout << std::fixed << std::setprecision(3);
    std::cout << "H2D_INT64_PAGEABLE_MS=" << h2d_int64_ms << "\n";
    std::cout << "H2D_XTOK32_PINNED_MS=" << h2d_xtok32_ms << "\n";
    std::cout << "K0_DENSE_GPU_MS=" << k0_ms << "\n";
    std::cout << "K1_NAIVE_GLOBAL_ATOMIC_MS=" << k1_ms << "\n";
    std::cout << "K2_SHARED_ATOMIC_MS=" << k2_ms << "\n";
    std::cout << "K3_GNCX_WARP_OPTIMAL_MS=" << k3_ms << "\n";
    std::cout << "SPEEDUP_K0_VS_K3=" << (k0_ms / k3_ms) << "x\n";
    std::cout << "SPEEDUP_K1_VS_K3=" << (k1_ms / k3_ms) << "x\n";
    return 0;
}


In [ ]:
# Compila para Tesla T4 (sm_75), executa e plota os logs!
import subprocess, re
import matplotlib.pyplot as plt

subprocess.run(['nvcc', '-O3', '-arch=sm_75', 'gncx_t4_colab.cu', '-o', 'gncx_t4_colab'], check=True)
res = subprocess.run(['./gncx_t4_colab'], capture_output=True, text=True, check=True)
print(res.stdout)

vals = {k: float(v) for k, v in re.findall(r'([A-Z0-9_]+)=([0-9.]+)', res.stdout)}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.bar(['H2D int64\n(Pageable 128MB)', 'H2D XToken32\n(Pinned 64MB)', 'Kernel K3\n(VRAM Compute)'],
        [vals['H2D_INT64_PAGEABLE_MS'], vals['H2D_XTOK32_PINNED_MS'], vals['K3_GNCX_WARP_OPTIMAL_MS']],
        color=['#ef4444', '#0ea5e9', '#22c55e'])
ax1.set_title('Gargalo 1: Barreira PCIe H2D vs Compute K3 (ms)')
ax1.set_ylabel('Tempo (ms) — Menor é Melhor')

ax2.bar(['K1: Global\nAtomic (10 bins)', 'K0: Dense\nBaseline', 'K2: Shared\nAtomic', 'K3: GNC-X\nWarp-Shuffle'],
        [vals['K1_NAIVE_GLOBAL_ATOMIC_MS'], vals['K0_DENSE_GPU_MS'], vals['K2_SHARED_ATOMIC_MS'], vals['K3_GNCX_WARP_OPTIMAL_MS']],
        color=['#ef4444', '#f97316', '#eab308', '#22c55e'])
ax2.set_title('Gargalo 2: Contenção Atômica nos 10 Crivos vs Warp-Shuffle (ms)')
ax2.set_ylabel('Tempo (ms) — Menor é Melhor')
plt.tight_layout()
plt.savefig('gncx_t4_colab_results.png', dpi=150)
plt.show()
